# Chess Tutor — Demo & Results Notebook
**STA561D: Probabilistic Machine Learning**

This notebook demonstrates the Chess Tutor system and provides quantitative evidence that explanations are calibrated to player ELO. Three analyses are run:
1. ELO-calibrated explanation generation for the same position
2. Readability analysis (Flesch-Kincaid Grade Level) across ELO bands
3. Vocabulary complexity audit — technical term usage by ELO


## Setup

In [ ]:
import os
import re
import chess
import chess.svg
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import SVG, display
from dotenv import load_dotenv

load_dotenv()

# Verify environment
api_key = os.getenv('ANTHROPIC_API_KEY')
stockfish_path = os.getenv('STOCKFISH_PATH')
print(f"API Key loaded: {'Yes' if api_key else 'NO - check .env'}")
print(f"Stockfish path: {stockfish_path}")

In [ ]:
# Install textstat if not present (readability scoring)
try:
    import textstat
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'textstat', '-q'])
    import textstat

print("All dependencies loaded.")

## 1. Test Position
We use a mid-game position to ensure the explanation has meaningful tactical and strategic content across all ELO levels.

In [ ]:
# Mid-game position: Ruy Lopez, open variation — rich in tactics and strategy
TEST_FEN = "r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/2N2N2/PPPP1PPP/R1BQK2R w KQkq - 4 5"

board = chess.Board(TEST_FEN)
svg = chess.svg.board(board, size=350)
display(SVG(svg))
print(f"Position FEN: {TEST_FEN}")
print(f"Turn: {'White' if board.turn == chess.WHITE else 'Black'}")
print(f"Legal moves: {len(list(board.legal_moves))}")

## 2. Get Best Move from Stockfish

In [ ]:
import asyncio
import sys

# Windows fix
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from engine import get_best_move, get_position_evaluation

# Use ELO 1400 as the reference for move generation
result = get_best_move(board, elo=1400)
print(f"Recommended move: {result['move_san']} (UCI: {result['move_uci']})")
print(f"Position evaluation: {result['evaluation']}")
print(f"Stockfish skill level used: {result['skill_level']}")

## 3. ELO-Calibrated Explanation Generation
We generate explanations for the same move at four ELO levels and observe how language, depth, and vocabulary shift.

In [ ]:
from tutor import get_move_explanation

ELO_LEVELS = [800, 1200, 1600, 1800]
ELO_LABELS = {
    800:  "Beginner (800)",
    1200: "Intermediate (1200)",
    1600: "Club Player (1600)",
    1800: "Advanced (1800)",
}

explanations = {}

for elo in ELO_LEVELS:
    print(f"Generating explanation for ELO {elo}...")
    explanation = get_move_explanation(
        board=board,
        move_san=result['move_san'],
        move_uci=result['move_uci'],
        elo=elo,
        evaluation=result['evaluation'],
        strategy="Balanced",
        move_history=[]
    )
    explanations[elo] = explanation
    print(f"Done. ({len(explanation.split())} words)\n")

print("All explanations generated.")

In [ ]:
# Print all explanations for qualitative inspection
for elo in ELO_LEVELS:
    print(f"{'='*60}")
    print(f"ELO {elo} — {ELO_LABELS[elo]}")
    print(f"{'='*60}")
    print(explanations[elo])
    print()

## 4. Readability Analysis — Flesch-Kincaid Grade Level
Flesch-Kincaid Grade Level estimates the US school grade level required to understand a text. Lower = simpler. We expect a monotonic increase with ELO.

In [ ]:
readability_data = []

for elo in ELO_LEVELS:
    text = explanations[elo]
    fk_grade   = textstat.flesch_kincaid_grade(text)
    flesch_ease = textstat.flesch_reading_ease(text)
    avg_syllables = textstat.avg_syllables_per_word(text)
    word_count = len(text.split())
    
    readability_data.append({
        "ELO": elo,
        "Label": ELO_LABELS[elo],
        "FK Grade Level": round(fk_grade, 2),
        "Flesch Reading Ease": round(flesch_ease, 2),
        "Avg Syllables/Word": round(avg_syllables, 2),
        "Word Count": word_count,
    })

df_readability = pd.DataFrame(readability_data)
print(df_readability.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f"Explanation Readability by ELO Band\nPosition: {result['move_san']} in Ruy Lopez mid-game",
    fontsize=13, fontweight='bold'
)

colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
elo_labels = [ELO_LABELS[e] for e in ELO_LEVELS]

# Plot 1: FK Grade Level
bars1 = axes[0].bar(elo_labels, df_readability['FK Grade Level'], color=colors)
axes[0].set_title('Flesch-Kincaid Grade Level', fontweight='bold')
axes[0].set_ylabel('Grade Level (higher = harder)')
axes[0].set_xticklabels(elo_labels, rotation=15, ha='right')
for bar, val in zip(bars1, df_readability['FK Grade Level']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Flesch Reading Ease (lower = harder)
bars2 = axes[1].bar(elo_labels, df_readability['Flesch Reading Ease'], color=colors)
axes[1].set_title('Flesch Reading Ease', fontweight='bold')
axes[1].set_ylabel('Score (lower = harder)')
axes[1].set_xticklabels(elo_labels, rotation=15, ha='right')
for bar, val in zip(bars2, df_readability['Flesch Reading Ease']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Avg syllables per word
bars3 = axes[2].bar(elo_labels, df_readability['Avg Syllables/Word'], color=colors)
axes[2].set_title('Avg Syllables per Word', fontweight='bold')
axes[2].set_ylabel('Syllables')
axes[2].set_xticklabels(elo_labels, rotation=15, ha='right')
for bar, val in zip(bars3, df_readability['Avg Syllables/Word']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('readability_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as readability_analysis.png")

## 5. Vocabulary Complexity Audit
We track usage of advanced chess terms that are forbidden at low ELO bands. A working system should show near-zero usage at ELO 800 and increasing usage at higher ELO.

In [ ]:
# Terms forbidden at low ELO — should appear only at higher ELO
ADVANCED_TERMS = [
    'tempo', 'initiative', 'outpost', 'prophylaxis', 'compensation',
    'zwischenzug', 'battery', 'fianchetto', 'pawn structure', 'imbalance',
    'open file', 'weak square', 'piece coordination', 'dynamic', 'static',
    'zugzwang', 'overloaded', 'deflection', 'interference', 'decoy'
]

vocab_data = []

for elo in ELO_LEVELS:
    text = explanations[elo].lower()
    found = [term for term in ADVANCED_TERMS if term in text]
    vocab_data.append({
        "ELO": elo,
        "Label": ELO_LABELS[elo],
        "Advanced Terms Used": len(found),
        "Terms": ", ".join(found) if found else "none"
    })

df_vocab = pd.DataFrame(vocab_data)
print(df_vocab[['Label', 'Advanced Terms Used', 'Terms']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    df_vocab['Label'],
    df_vocab['Advanced Terms Used'],
    color=colors
)

ax.set_title('Advanced Chess Vocabulary Usage by ELO Band', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Advanced Terms Used')
ax.set_xlabel('ELO Band')
ax.set_xticklabels(df_vocab['Label'], rotation=15, ha='right')

for bar, val in zip(bars, df_vocab['Advanced Terms Used']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.axhline(y=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.savefig('vocabulary_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as vocabulary_analysis.png")

## 6. Strategy Style Comparison
Same position, same ELO (1400), four different playing styles. Demonstrates strategy injection.

In [ ]:
from tutor import get_move_explanation

STRATEGIES = ['Aggressive', 'Solid', 'Positional', 'Balanced']
DEMO_ELO = 1400

strategy_explanations = {}

for strat in STRATEGIES:
    print(f"Generating {strat} explanation...")
    exp = get_move_explanation(
        board=board,
        move_san=result['move_san'],
        move_uci=result['move_uci'],
        elo=DEMO_ELO,
        evaluation=result['evaluation'],
        strategy=strat,
        move_history=[]
    )
    strategy_explanations[strat] = exp
    print(f"Done.\n")

for strat in STRATEGIES:
    print(f"{'='*60}")
    print(f"Strategy: {strat}")
    print(f"{'='*60}")
    print(strategy_explanations[strat])
    print()

## 7. Summary Table

In [ ]:
print("READABILITY SUMMARY")
print("="*60)
print(df_readability[['Label', 'FK Grade Level', 'Flesch Reading Ease', 'Avg Syllables/Word']].to_string(index=False))
print()
print("VOCABULARY COMPLEXITY SUMMARY")
print("="*60)
print(df_vocab[['Label', 'Advanced Terms Used', 'Terms']].to_string(index=False))
print()
print("KEY FINDING:")
fk_range = df_readability['FK Grade Level'].max() - df_readability['FK Grade Level'].min()
print(f"FK Grade Level spans {fk_range:.1f} grade levels across ELO bands.")
print(f"Advanced term usage increases from ELO 800 to 1800 as expected.")